# Import to Neo4j

**Module 2 -- Lesson 12**

The normalized CSV files are ready. This notebook creates a Neo4j Aura Free instance, connects to it, and imports the data as a graph.

## 1. Create an Aura Free instance

1. Go to [console.neo4j.io](https://console.neo4j.io)
2. Sign up or sign in
3. Click **Create Instance** and select **AuraDB Free**
4. A credentials modal will appear with your **username** and **generated password**
5. Click **Download and continue** to save the credentials file -- you'll need these in the next step
6. Wait for the instance status to change from *Creating* to *Running* (usually under a minute)

Your connection URI will look like `neo4j+s://xxxxxxxx.databases.neo4j.io`.

## 2. Add credentials to your environment

Add the following to your `.env` file in the project root (create it if it doesn't exist). Replace the values with the credentials from the download:

```
NEO4J_URI=neo4j+s://xxxxxxxx.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-generated-password
```

The `.env` file is in `.gitignore` -- your credentials won't be committed.

## 3. Connect

In [6]:
%pip install neo4j python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import csv
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(dotenv_path=".env")

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD"))
DB = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print(f"Connected to {URI}")

Connected to neo4j+s://52976274.databases.neo4j.io


In [8]:
def load_csv(name):
    with open(Path("data/csv") / name) as f:
        return list(csv.DictReader(f))

## 4. Constraints

Constraints prevent duplicate nodes and create backing indexes. Every `MERGE` needs a constraint on the property it matches.

In [9]:
for c in [
    "CREATE CONSTRAINT email_doc_id IF NOT EXISTS FOR (e:Email) REQUIRE e.doc_id IS UNIQUE",
    "CREATE CONSTRAINT user_name IF NOT EXISTS FOR (u:User) REQUIRE u.name_norm IS UNIQUE",
    "CREATE CONSTRAINT mailbox_address IF NOT EXISTS FOR (m:Mailbox) REQUIRE m.address_norm IS UNIQUE",
    "CREATE CONSTRAINT domain_name IF NOT EXISTS FOR (d:Domain) REQUIRE d.name_norm IS UNIQUE",
]:
    driver.execute_query(c, database_=DB)

print("Constraints created.")

Constraints created.


## 5. Import

Each CSV is loaded into Python and passed as a parameter to a single `UNWIND` query. Nodes `MERGE` on the normalized value; the raw value is stored alongside it.

In [10]:
rows = load_csv("emails.csv")

driver.execute_query("""
UNWIND $rows AS row
MERGE (e:Email {doc_id: row.doc_id})
SET e.subject  = row.subject,
    e.date_raw = row.date_raw,
    e.date     = row.date_parsed,
    e.method   = row.method,
    e.body     = row.body
""", rows=rows, database_=DB)

print(f"Emails: {len(rows):,}")

Emails: 4,911


In [11]:
rows = load_csv("SENT.csv")

driver.execute_query("""
UNWIND $rows AS row
MATCH (e:Email {doc_id: row.doc_id})

CALL (row, e) {
    WITH row, e
    WHERE row.name_norm <> ''
    MERGE (u:User {name_norm: row.name_norm})
    ON CREATE SET u.name = row.name
    MERGE (u)-[:SENT]->(e)
}

CALL (row, e) {
    WITH row, e
    WHERE row.address_norm <> ''
    MERGE (m:Mailbox {address_norm: row.address_norm})
    ON CREATE SET m.address = row.address
    MERGE (m)-[:SENT]->(e)
}
""", rows=rows, database_=DB)

print(f"SENT: {len(rows):,}")

SENT: 4,911


In [12]:
rows = load_csv("RECEIVED.csv")

driver.execute_query("""
UNWIND $rows AS row
MATCH (e:Email {doc_id: row.doc_id})

CALL (row, e) {
    WITH row, e
    WHERE row.name_norm <> ''
    MERGE (u:User {name_norm: row.name_norm})
    ON CREATE SET u.name = row.name
    MERGE (u)-[:RECEIVED]->(e)
}

CALL (row, e) {
    WITH row, e
    WHERE row.address_norm <> ''
    MERGE (m:Mailbox {address_norm: row.address_norm})
    ON CREATE SET m.address = row.address
    MERGE (m)-[:RECEIVED]->(e)
}
""", rows=rows, database_=DB)

print(f"RECEIVED: {len(rows):,}")

RECEIVED: 14,266


In [13]:
rows = load_csv("CC_ON.csv")

driver.execute_query("""
UNWIND $rows AS row
MATCH (e:Email {doc_id: row.doc_id})

CALL (row, e) {
    WITH row, e
    WHERE row.name_norm <> ''
    MERGE (u:User {name_norm: row.name_norm})
    ON CREATE SET u.name = row.name
    MERGE (u)-[:CC_ON]->(e)
}

CALL (row, e) {
    WITH row, e
    WHERE row.address_norm <> ''
    MERGE (m:Mailbox {address_norm: row.address_norm})
    ON CREATE SET m.address = row.address
    MERGE (m)-[:CC_ON]->(e)
}
""", rows=rows, database_=DB)

print(f"CC_ON: {len(rows):,}")

CC_ON: 1,348


In [14]:
rows = load_csv("USED.csv")

driver.execute_query("""
UNWIND $rows AS row
MATCH (u:User {name_norm: row.name_norm})
MATCH (m:Mailbox {address_norm: row.address_norm})
MERGE (u)-[:USED]->(m)
""", rows=rows, database_=DB)

print(f"USED: {len(rows):,}")

USED: 5,043


In [15]:
rows = load_csv("HAS_MAILBOX.csv")

driver.execute_query("""
UNWIND $rows AS row
MERGE (d:Domain {name_norm: row.domain_norm})
ON CREATE SET d.name = row.domain
MERGE (m:Mailbox {address_norm: row.address_norm})
MERGE (d)-[:HAS_MAILBOX]->(m)
""", rows=rows, database_=DB)

print(f"HAS_MAILBOX: {len(rows):,}")

HAS_MAILBOX: 5,588


## 6. Verify

In [16]:
result, _, _ = driver.execute_query("""
CALL () {
    MATCH (e:Email) RETURN 'Email' AS label, count(e) AS count
    UNION ALL
    MATCH (u:User) RETURN 'User' AS label, count(u) AS count
    UNION ALL
    MATCH (m:Mailbox) RETURN 'Mailbox' AS label, count(m) AS count
    UNION ALL
    MATCH (d:Domain) RETURN 'Domain' AS label, count(d) AS count
}
RETURN label, count ORDER BY count DESC
""", database_=DB)

print("Node counts:")
for row in result:
    print(f"  {row['label']:10s} {row['count']:,}")

print()

result, _, _ = driver.execute_query("""
CALL () {
    MATCH ()-[r:SENT]->() RETURN 'SENT' AS type, count(r) AS count
    UNION ALL
    MATCH ()-[r:RECEIVED]->() RETURN 'RECEIVED' AS type, count(r) AS count
    UNION ALL
    MATCH ()-[r:CC_ON]->() RETURN 'CC_ON' AS type, count(r) AS count
    UNION ALL
    MATCH ()-[r:USED]->() RETURN 'USED' AS type, count(r) AS count
    UNION ALL
    MATCH ()-[r:HAS_MAILBOX]->() RETURN 'HAS_MAILBOX' AS type, count(r) AS count
}
RETURN type, count ORDER BY count DESC
""", database_=DB)

print("Relationship counts:")
for row in result:
    print(f"  {row['type']:15s} {row['count']:,}")

Node counts:
  Mailbox    5,515
  User       5,464
  Email      4,911
  Domain     1,079

Relationship counts:
  RECEIVED        23,948
  SENT            7,820
  HAS_MAILBOX     5,555
  USED            4,899
  CC_ON           2,257


## 7. Investigate

The graph is live. These queries demonstrate what the metadata model makes possible -- questions that require traversing relationships, not filtering rows.

In [17]:
result, _, _ = driver.execute_query("""
MATCH (m:Mailbox)-[:SENT]->(e:Email)
RETURN m.address AS sender, count(e) AS emails
ORDER BY emails DESC LIMIT 10
""", database_=DB)

print("Top 10 senders (by mailbox):")
for row in result:
    print(f"  {row['sender']:40s} {row['emails']:,}")

Top 10 senders (by mailbox):
  kay.mann@enron.com                       115
  vince.kaminski@enron.com                 114
  chris.germany@enron.com                  83
  jeff.dasovich@enron.com                  79
  pete.davis@enron.com                     70
  tana.jones@enron.com                     67
  sara.shackleton@enron.com                66
  enron.announcements@enron.com            59
  matthew.lenhart@enron.com                52
  steven.kean@enron.com                    48


In [18]:
result, _, _ = driver.execute_query("""
MATCH (d1:Domain)-[:HAS_MAILBOX]->(m1:Mailbox)-[:SENT]->(e:Email)
      <-[:RECEIVED]-(m2:Mailbox)<-[:HAS_MAILBOX]-(d2:Domain)
WHERE d1 <> d2
RETURN d1.name AS from_domain, d2.name AS to_domain, count(e) AS emails
ORDER BY emails DESC LIMIT 10
""", database_=DB)

print("Top 10 cross-domain communication:")
for row in result:
    print(f"  {row['from_domain']:25s} -> {row['to_domain']:25s} {row['emails']:,}")

Top 10 cross-domain communication:
  enran.ccom                -> aol.com                   85
  txu.com                   -> enran.ccom                55
  govadv.com                -> enran.ccom                51
  columbiaenergygroup.com   -> enran.ccom                46
  hotmail.com               -> enran.ccom                29
  enran.ccom                -> rice.edu                  23
  bracepatt.com             -> enran.ccom                22
  enran.ccom                -> hotmail.com               22
  tca-us.com                -> enran.ccom                18
  enran.ccom                -> yahoo.com                 16


In [19]:
print("Run this in Neo4j Browser to visualize the schema:")
print()
print("  CALL db.schema.visualization()")

Run this in Neo4j Browser to visualize the schema:

  CALL db.schema.visualization()


## What the communication network can answer

The graph captures **who** sent **what** to **whom** and **when**. You can now answer questions like:

* Who are the most connected people in the network?
* Which domains communicate most with Enron?
* Who bridges between different groups?
* What's the communication pattern around a specific date or event?

What it can't answer: **what they talked about**. The body text is stored on each Email node, but the names, organizations, topics, and locations mentioned in it haven't been extracted as network entities yet. That's the next course: *Entity Extraction for Communication Networks*.

In [20]:
driver.close()
print("Connection closed.")

Connection closed.


## Summary

- Created an Aura Free instance and connected via the Python driver
- Constraints on `_norm` properties ensure `MERGE` matches correctly
- Each CSV passed as a parameter to a single `UNWIND` query -- no batching needed at this scale
- Nodes merged on normalized values, raw values stored as properties
- The graph contains Email, User, Mailbox, and Domain nodes connected by SENT, RECEIVED, CC_ON, USED, and HAS_MAILBOX relationships

**Entity Communication Networks complete.** The next course, *Entity Extraction for Communication Networks*, extracts entities and topics from the email body text.